In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
from pathlib import Path
import sys

working_directory = Path.cwd().resolve()
PROJECT_ROOT = next(
    (path for path in (working_directory, *working_directory.parents)
     if (path / "pyproject.toml").is_file() and (path / "src").is_dir()),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError("Open this notebook from within the project directory.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
    


In [4]:
from pathlib import Path

import numpy as np
import pandas as pd

from src.data.loader import load_dataset
from src.data.windows import make_rolling_windows
from src.features.extractor import extract_window_features

import warnings
warnings.filterwarnings('ignore')

In [5]:
TSFM_MODELS = ["chronos2", "moirai2", "timesfm3"]
PREDICTION_LENGTHS = [24, 96, 192]

### **Feature Extraction**

In [6]:
dataset = load_dataset('ETTh1')
rows = []

In [10]:
for prediction_length in PREDICTION_LENGTHS:
    windows = make_rolling_windows(
        dataset=dataset,
        context_length=512,
        prediction_length=prediction_length,
        n_windows=20,
        stride=prediction_length, 
        columns=["OT"])
    
    for window in windows:
        features = extract_window_features(
            window=window, 
            seasonal_period=dataset.seasonal_period)
        rows.append({
            "dataset": dataset.name,
            "target": "OT",
            "window_id": window.window_id,
            "prediction_length": prediction_length})

In [11]:
features_df = pd.DataFrame(rows)
features_df

,dataset,target,window_id,prediction_length
0,ETTh1,OT,0,24
1,ETTh1,OT,1,24
2,ETTh1,OT,2,24
3,ETTh1,OT,3,24
4,ETTh1,OT,4,24
5,ETTh1,OT,5,24
6,ETTh1,OT,6,24
7,ETTh1,OT,7,24
8,ETTh1,OT,8,24
9,ETTh1,OT,9,24


### **Evaluation Metrics**

In [13]:
metrics_dir = PROJECT_ROOT / "results" / "pilot" / "metrics"
frames = []

for model in TSFM_MODELS + ["seasonal_naive"]:
    path = metrics_dir / f"{model}_ETTh1.parquet"
    frame = pd.read_parquet(path)
    # Older result files mix microsecond and nanosecond timestamps.
    frame["cutoff_timestamp"] = frame["cutoff_timestamp"].dt.as_unit("ns")
    frames.append(frame)
    
metrics_df = pd.concat(frames, ignore_index=True)
metrics_df

,dataset,model,target,window_id,cutoff_timestamp,context_length,prediction_length,mae,rmse,smape,mase,inference_time_seconds
0,ETTh1,chronos2,OT,0,2018-06-06 19:00:00,512,24,0.404364,0.490909,3.594163,0.302008,0.057224
1,ETTh1,chronos2,OT,1,2018-06-07 19:00:00,512,24,0.930147,1.242307,8.732762,0.725469,0.057433
2,ETTh1,chronos2,OT,2,2018-06-08 19:00:00,512,24,1.929759,2.791145,26.035334,1.734458,0.093347
3,ETTh1,chronos2,OT,3,2018-06-09 19:00:00,512,24,0.996997,1.098213,12.755473,0.853528,0.060495
4,ETTh1,chronos2,OT,4,2018-06-10 19:00:00,512,24,0.677860,0.873132,7.021194,0.557674,0.061168
...,...,...,...,...,...,...,...,...,...,...,...,...
235,ETTh1,seasonal_naive,OT,15,2018-05-17 19:00:00,512,192,5.128016,5.868920,47.992767,2.860091,0.000040
236,ETTh1,seasonal_naive,OT,16,2018-05-25 19:00:00,512,192,1.573307,1.969959,16.867500,0.858638,0.000044
237,ETTh1,seasonal_naive,OT,17,2018-06-02 19:00:00,512,192,1.275391,1.728358,13.764468,0.824011,0.000050
238,ETTh1,seasonal_naive,OT,18,2018-06-10 19:00:00,512,192,2.388120,2.697654,25.750809,1.964702,0.000043


### **Merge Features and Metrics & Compute Margins**

In [18]:
key = ["dataset", "target", "window_id", "prediction_length"]
mase = metrics_df.pivot(index=key, columns="model", values="mase").reset_index()
mase = mase.rename(
    columns={
        "chronos2": "chronos2_mase",
        "moirai2": "moirai2_mase",
        "timesfm3": "timesfm3_mase",
        "seasonal_naive": "seasonal_naive_mase"})

In [19]:
meta = features_df.merge(mase, on=key, how="inner", validate="one_to_one")
tsfm_columns = ["chronos2_mase", "moirai2_mase", "timesfm3_mase"]
model_names = np.array(["chronos2", "moirai2", "timesfm3"])

In [20]:
scores = meta[tsfm_columns].to_numpy()
order = np.argsort(scores, axis=1)

In [21]:
meta["best_tsfm"] = model_names[order[:, 0]]

In [22]:
best = np.take_along_axis(scores, order[:, :1], axis=1).ravel()
second_best = np.take_along_axis(scores, order[:, 1: 2], axis=1).ravel()

In [23]:
meta["winner_margin"] = second_best - best
meta["winner_margin_relative"] = meta["winner_margin"] / np.maximum(best, 1e-8)

In [24]:
meta

,dataset,target,window_id,prediction_length,chronos2_mase,moirai2_mase,seasonal_naive_mase,timesfm3_mase,best_tsfm,winner_margin,winner_margin_relative
0,ETTh1,OT,0,24,0.302008,0.379346,0.888932,0.418249,chronos2,0.077339,0.256081
1,ETTh1,OT,1,24,0.725469,0.688865,0.887000,0.772441,moirai2,0.036605,0.053137
2,ETTh1,OT,2,24,1.734458,1.762020,2.534228,1.844116,chronos2,0.027561,0.015890
3,ETTh1,OT,3,24,0.853528,0.583212,1.864404,0.733799,moirai2,0.150587,0.258203
4,ETTh1,OT,4,24,0.557674,0.489419,1.065874,0.589102,moirai2,0.068255,0.139461
5,ETTh1,OT,5,24,0.616770,0.553078,1.322687,0.724938,moirai2,0.063692,0.115160
6,ETTh1,OT,6,24,0.567623,0.392172,0.777660,0.788199,moirai2,0.175451,0.447382
7,ETTh1,OT,7,24,0.420152,0.281050,0.334523,0.305396,moirai2,0.024346,0.086623
8,ETTh1,OT,8,24,0.293693,0.332797,0.590754,0.295289,chronos2,0.001596,0.005435
9,ETTh1,OT,9,24,0.747224,1.019976,1.448387,0.809545,chronos2,0.062321,0.083403


In [25]:
output_dir = PROJECT_ROOT / "results" / "meta_dataset"
output_dir.mkdir(parents=True, exist_ok=True)
output_path = output_dir / "ETTh1.parquet"

In [27]:
meta.to_parquet(output_path, index=False)
print(f"metadata of ETTh1 saved to {output_path.relative_to(PROJECT_ROOT).as_posix()}")

metadata of ETTh1 saved to results/meta_dataset/ETTh1.parquet


In [28]:
print(f"Shape: {meta.shape}")
print("Best TSFM counts:")
print(meta["best_tsfm"].value_counts())

Shape: (60, 11)
Best TSFM counts:
best_tsfm
chronos2    23
moirai2     21
timesfm3    16
Name: count, dtype: int64
